# 🚀 GitHub Repository Migration Tool
### Migrate repos from Account 1 → Account 2 with clone, push & delete
---
> **Workflow**: List repos → Select → Clone locally → Push to Account 2 → (Optional) Delete from Account 1

## 📦 Cell 1 — Install Dependencies

In [7]:
# Install required libraries
#!pip install PyGithub requests tqdm colorama tabulate -q
#print('✅ Dependencies installed.')

## ⚙️ Cell 2 — Configuration

Fill in your credentials below.

**How to get a Personal Access Token (PAT):**
1. Go to GitHub → Settings → Developer Settings → Personal Access Tokens → Tokens (classic)
2. Generate new token with scopes: `repo`, `delete_repo`
3. Paste the token below

> ⚠️ **Never commit this notebook with real tokens to any public repo!**

In [ ]:
import os

# ─────────────────────────────────────────────
# ACCOUNT 1 — Source (repos will be migrated FROM here)
# ─────────────────────────────────────────────
ACCOUNT1_TOKEN    = ""   # PAT for Account 1 (needs: repo, delete_repo)
ACCOUNT1_USERNAME = ""

# ─────────────────────────────────────────────
# ACCOUNT 2 — Destination (repos will be pushed TO here)
# ─────────────────────────────────────────────
ACCOUNT2_TOKEN    = ""   # PAT for Account 2 (needs: repo)
ACCOUNT2_USERNAME = ""

# ─────────────────────────────────────────────
# LOCAL CLONE DIRECTORY
# ─────────────────────────────────────────────
CLONE_BASE_DIR = os.path.expanduser("D:\\REPOS\\github_migration")   # Change if needed

# Safety flag: Set to True only when you're ready to delete from Account 1
ALLOW_DELETE = True

print('✅ Configuration set.')
print(f'   Clone directory : {CLONE_BASE_DIR}')
print(f'   Source account  : {ACCOUNT1_USERNAME}')
print(f'   Target account  : {ACCOUNT2_USERNAME}')
print(f'   Delete allowed  : {ALLOW_DELETE}')

✅ Configuration set.
   Clone directory : D:\REPOS\github_migration
   Source account  : Rajbeer-singh2706
   Target account  : testsparkaws
   Delete allowed  : True


## 🔌 Cell 3 — Connect to GitHub & Imports

In [27]:
import os
import subprocess
import shutil
import json
from pathlib import Path
from datetime import datetime

from github import Github, GithubException
from tqdm.notebook import tqdm
from tabulate import tabulate
from colorama import Fore, Style, init
init(autoreset=True)

# Connect to both GitHub accounts
g1 = Github(ACCOUNT1_TOKEN)
g2 = Github(ACCOUNT2_TOKEN)

user1 = g1.get_user()
user2 = g2.get_user()

print(f'✅ Connected to Account 1 : {user1.login} ({user1.name})')
print(f'✅ Connected to Account 2 : {user2.login} ({user2.name})')

# Create clone directory
Path(CLONE_BASE_DIR).mkdir(parents=True, exist_ok=True)
print(f'✅ Clone directory ready  : {CLONE_BASE_DIR}')

C:\Users\comp n\AppData\Local\Temp\ipykernel_18940\2743901981.py:15: DeprecationWarning: Argument login_or_token is deprecated, please use auth=github.Auth.Token(...) instead
  g1 = Github(ACCOUNT1_TOKEN)
C:\Users\comp n\AppData\Local\Temp\ipykernel_18940\2743901981.py:16: DeprecationWarning: Argument login_or_token is deprecated, please use auth=github.Auth.Token(...) instead
  g2 = Github(ACCOUNT2_TOKEN)


✅ Connected to Account 1 : Rajbeer-singh2706 (None)
✅ Connected to Account 2 : testsparkaws (None)
✅ Clone directory ready  : D:\REPOS\github_migration


## 📋 Cell 4 — List All Repos from Account 1

In [33]:
print('⏳ Fetching repositories from Account 1...\n')

repos_raw = list(user1.get_repos(type='owner'))  # Only repos owned by user

repo_data = []
repo_name = []
for r in repos_raw:
    if "mastering" in r.name:
        repo_name.append(r.name)
    repo_data.append({
        'name'       : r.name,
        'private'    : '🔒 Private' if r.private else '🌐 Public',
        'size_kb'    : r.size,
        'stars'      : r.stargazers_count,
        'forks'      : r.forks_count,
        'updated'    : r.updated_at.strftime('%Y-%m-%d'),
        'description': (r.description or '')[:50],
        'clone_url'  : r.clone_url,
        'default_branch': r.default_branch,
    })

# Display table
table_rows = [
    [i+1, d['name'], d['private'], f"{d['size_kb']:,} KB", d['stars'], d['updated'], d['description']]
    for i, d in enumerate(repo_data)
]
headers = ['#', 'Repo Name', 'Visibility', 'Size', '⭐', 'Updated', 'Description']

#print(repo_name)
print(tabulate(table_rows, headers=headers, tablefmt='rounded_outline'))
print(f'\n📦 Total repositories found: {len(repo_data)}')

⏳ Fetching repositories from Account 1...

╭─────┬─────────────────────────────┬──────────────┬──────────┬──────┬────────────┬─────────────────────────────╮
│   # │ Repo Name                   │ Visibility   │ Size     │   ⭐ │ Updated    │ Description                 │
├─────┼─────────────────────────────┼──────────────┼──────────┼──────┼────────────┼─────────────────────────────┤
│   1 │ agentic-rag-project         │ 🔒 Private   │ 9,636 KB │    0 │ 2026-05-18 │ agentic-rag-project         │
│   2 │ agentic-trading-bot-project │ 🌐 Public    │ 1,210 KB │    0 │ 2026-05-13 │ agentic-trading-bot-project │
│   3 │ data-governance-copilot     │ 🌐 Public    │ 969 KB   │    0 │ 2026-05-18 │ data-governance-copilot     │
│   4 │ DocManager                  │ 🌐 Public    │ 7,490 KB │    0 │ 2026-03-31 │ DocManager                  │
│   5 │ streamforge-de-project      │ 🌐 Public    │ 37 KB    │    1 │ 2026-05-18 │ streamforge-de-project      │
│   6 │ vedicOs                     │ 🔒 Private   │

## 🔍 Cell 5 — Filter / Select Repos to Migrate
Choose which repos to migrate. Three modes:
- `'all'` — migrate every repo
- `'include'` — only the listed names
- `'exclude'` — all except the listed names

In [29]:
# ─────────────────────────────────────────────
# FILTER MODE: 'all' | 'include' | 'exclude'
# ─────────────────────────────────────────────
FILTER_MODE = 'exclude'

# Used when FILTER_MODE = 'include' or 'exclude'
REPO_LIST = ['data-governance-copilot','agentic-rag-project','streamforge-de-project','DocManager','vedicOs']

# ── Apply filter ──────────────────────────────
if FILTER_MODE == 'all':
    selected_repos = repo_data
elif FILTER_MODE == 'include':
    selected_repos = [r for r in repo_data if r['name'] in REPO_LIST]
elif FILTER_MODE == 'exclude':
    selected_repos = [r for r in repo_data if r['name'] not in REPO_LIST]
else:
    raise ValueError("FILTER_MODE must be 'all', 'include', or 'exclude'")

print(f'✅ Filter mode   : {FILTER_MODE}')
print(f'✅ Repos selected: {len(selected_repos)} / {len(repo_data)}')
for r in selected_repos:
    print(f'   → {r["name"]} ({r["private"]})')

✅ Filter mode   : exclude
✅ Repos selected: 40 / 45
   → 1stproject-document-portal (🔒 Private)
   → 2ndProject_ECommerce_Product_Assistant (🔒 Private)
   → advance_nlp_generative_ai_bootcamp (🔒 Private)
   → agentic-ai-project (🌐 Public)
   → agentic-ai-resource (🔒 Private)
   → agentic-trading-bot-project (🌐 Public)


   → ai-learning-with-claude (🌐 Public)
   → ai-research-assistant (🌐 Public)
   → assignment1-encoding-implementation (🌐 Public)
   → aws-agentic-ai (🔒 Private)
   → aws-certification (🌐 Public)
   → aws-data-enginner (🌐 Public)
   → comp-specific-question (🌐 Public)
   → course3-statistics-essentials (🔒 Private)
   → course6-Machine-Learning-2 (🔒 Private)
   → custmor_support_system (🔒 Private)
   → customer_support_system (🌐 Public)
   → de-project (🌐 Public)
   → ecomm-prod-assistant-project (🔒 Private)
   → first-aws-de-batch (🔒 Private)
   → genai-application-development (🔒 Private)
   → gesture-recognition-project (🌐 Public)
   → howtobecomedataarchitect-repo (🔒 Private)
   → inter-question (🌐 Public)
   → langchain_langsmith (🌐 Public)
   → learning-generative-ai (🌐 Public)
   → llmops (🔒 Private)
   → mastering-aws-redshift (🔒 Private)
   → ml-ai-course-repo (🌐 Public)
   → modern-route-genai-agenticai (🌐 Public)
   → module16-project (🌐 Public)
   → my-course (🔒 Private)
   →

## 🛠️ Cell 6 — Helper Functions

In [30]:
def run_cmd(cmd, cwd=None):
    """Run a shell command and return (success, stdout, stderr)."""
    result = subprocess.run(
        cmd, shell=True, cwd=cwd,
        capture_output=True, text=True
    )
    return result.returncode == 0, result.stdout.strip(), result.stderr.strip()


def clone_repo(repo_info, target_dir):
    """Clone a repo from Account 1 using token-embedded URL."""
    repo_name = repo_info['name']
    clone_path = os.path.join(target_dir, repo_name)

    # Remove if already exists
    if os.path.exists(clone_path):
        shutil.rmtree(clone_path)

    # Embed token for HTTPS authentication
    auth_url = f"https://{ACCOUNT1_TOKEN}@github.com/{ACCOUNT1_USERNAME}/{repo_name}.git"
    ok, out, err = run_cmd(f'git clone --mirror "{auth_url}" "{clone_path}"')

    if not ok:
        raise RuntimeError(f'Clone failed for {repo_name}: {err}')
    return clone_path


def create_repo_on_account2(repo_info):
    """Create repo on Account 2 with same visibility as Account 1."""
    name    = repo_info['name']
    private = repo_info['private'] == '🔒 Private'
    desc    = repo_info['description']

    try:
        existing = g2.get_repo(f"{ACCOUNT2_USERNAME}/{name}")
        print(f'   ⚠️  Repo already exists on Account 2: {name} (will push to it)')
        return existing
    except GithubException:
        pass  # Doesn't exist, create it

    new_repo = user2.create_repo(
        name        = name,
        description = desc,
        private     = private,
        auto_init   = False,
    )
    return new_repo


def push_to_account2(clone_path, repo_info):
    """Push all refs (branches + tags) to Account 2."""
    repo_name  = repo_info['name']
    remote_url = f"https://{ACCOUNT2_TOKEN}@github.com/{ACCOUNT2_USERNAME}/{repo_name}.git"

    ok, out, err = run_cmd(f'git push --mirror "{remote_url}"', cwd=clone_path)
    if not ok:
        raise RuntimeError(f'Push failed for {repo_name}: {err}')


def delete_from_account1(repo_name):
    """Delete repo from Account 1 (irreversible!)."""
    if not ALLOW_DELETE:
        print(f'   ⏭️  Skipping delete (ALLOW_DELETE=False): {repo_name}')
        return False
    try:
        repo = g1.get_repo(f"{ACCOUNT1_USERNAME}/{repo_name}")
        repo.delete()
        return True
    except GithubException as e:
        print(f'   ❌ Could not delete {repo_name}: {e}')
        return False


print('✅ Helper functions loaded.')

✅ Helper functions loaded.


## 🧪 Cell 7 — Dry Run (Preview Only — No Changes Made)
Run this first to verify everything looks right before the real migration.

In [31]:
print('🔍 DRY RUN — No changes will be made\n')
print(f'  Source  : {ACCOUNT1_USERNAME} ({len(selected_repos)} repos)')
print(f'  Target  : {ACCOUNT2_USERNAME}')
print(f'  CloneDir: {CLONE_BASE_DIR}')
print(f'  Delete? : {ALLOW_DELETE}\n')

rows = []
for r in selected_repos:
    rows.append([
        r['name'],
        r['private'],
        f"{r['size_kb']:,} KB",
        '✅ Clone → Push → Delete' if ALLOW_DELETE else '✅ Clone → Push',
    ])

print(tabulate(rows, headers=['Repo', 'Visibility', 'Size', 'Actions'], tablefmt='rounded_outline'))
print(f'\n⚡ Ready to migrate {len(selected_repos)} repos. Run Cell 8 to start.')

🔍 DRY RUN — No changes will be made

  Source  : Rajbeer-singh2706 (40 repos)
  Target  : testsparkaws
  CloneDir: D:\REPOS\github_migration
  Delete? : True

╭────────────────────────────────────────┬──────────────┬────────────┬──────────────────────────╮
│ Repo                                   │ Visibility   │ Size       │ Actions                  │
├────────────────────────────────────────┼──────────────┼────────────┼──────────────────────────┤
│ 1stproject-document-portal             │ 🔒 Private   │ 8,513 KB   │ ✅ Clone → Push → Delete │
│ 2ndProject_ECommerce_Product_Assistant │ 🔒 Private   │ 5,408 KB   │ ✅ Clone → Push → Delete │
│ advance_nlp_generative_ai_bootcamp     │ 🔒 Private   │ 59,670 KB  │ ✅ Clone → Push → Delete │
│ agentic-ai-project                     │ 🌐 Public    │ 3 KB       │ ✅ Clone → Push → Delete │
│ agentic-ai-resource                    │ 🔒 Private   │ 130,555 KB │ ✅ Clone → Push → Delete │
│ agentic-trading-bot-project            │ 🌐 Public    │ 1,210 KB  

## 🚀 Cell 8 — RUN MIGRATION
> ⚠️ This clones, pushes, and optionally deletes repos. Review the dry run first!

In [32]:
migration_log = []  # Track results for summary

print(f'🚀 Starting migration of {len(selected_repos)} repos...\n')
print('=' * 60)

for i, repo_info in enumerate(selected_repos, 1):
    name   = repo_info['name']
    status = {'repo': name, 'clone': '❌', 'create': '❌', 'push': '❌', 'delete': 'N/A', 'error': ''}

    print(f'\n[{i}/{len(selected_repos)}] Processing: {name}')

    try:
        # ── Step 1: Clone ──────────────────────────────
        print(f'  📥 Cloning from Account 1...')
        clone_path = clone_repo(repo_info, CLONE_BASE_DIR)
        status['clone'] = '✅'
        print(f'  ✅ Cloned to: {clone_path}')

        # ── Step 2: Create repo on Account 2 ───────────
        print(f'  🏗️  Creating repo on Account 2...')
        new_repo = create_repo_on_account2(repo_info)
        status['create'] = '✅'
        print(f'  ✅ Repo ready: {new_repo.html_url}')

        # ── Step 3: Push mirror ─────────────────────────
        print(f'  📤 Pushing all branches & tags to Account 2...')
        push_to_account2(clone_path, repo_info)
        status['push'] = '✅'
        print(f'  ✅ Push complete.')

        # ── Step 4: Delete from Account 1 (optional) ───
        if ALLOW_DELETE:
            print(f'  🗑️  Deleting from Account 1...')
            deleted = delete_from_account1(name)
            status['delete'] = '✅' if deleted else '❌'
            print(f'  {"✅ Deleted." if deleted else "❌ Delete failed."}')
        else:
            status['delete'] = '⏭️ Skipped'

        # ── Clean up local clone ────────────────────────
        shutil.rmtree(clone_path)
        print(f'  🧹 Local clone cleaned up.')

    except Exception as e:
        status['error'] = str(e)
        print(f'  ❌ ERROR: {e}')

    migration_log.append(status)

print('\n' + '=' * 60)
print('✅ Migration complete!')

🚀 Starting migration of 40 repos...


[1/40] Processing: 1stproject-document-portal
  📥 Cloning from Account 1...
  ✅ Cloned to: D:\REPOS\github_migration\1stproject-document-portal
  🏗️  Creating repo on Account 2...
  ✅ Repo ready: https://github.com/testsparkaws/1stproject-document-portal
  📤 Pushing all branches & tags to Account 2...
  ✅ Push complete.
  🗑️  Deleting from Account 1...
  ✅ Deleted.
  ❌ ERROR: [WinError 5] Access is denied: 'D:\\REPOS\\github_migration\\1stproject-document-portal\\objects\\pack\\pack-9fcbb9c1eb7ba6f849c6de2deb6fcf3f8ae248ad.idx'

[2/40] Processing: 2ndProject_ECommerce_Product_Assistant
  📥 Cloning from Account 1...
  ✅ Cloned to: D:\REPOS\github_migration\2ndProject_ECommerce_Product_Assistant
  🏗️  Creating repo on Account 2...
  ✅ Repo ready: https://github.com/testsparkaws/2ndProject_ECommerce_Product_Assistant
  📤 Pushing all branches & tags to Account 2...
  ✅ Push complete.
  🗑️  Deleting from Account 1...
  ✅ Deleted.
  ❌ ERROR: [WinError 5]

## 📊 Cell 9 — Migration Summary Report

In [ ]:
print('\n📊 MIGRATION SUMMARY REPORT')
print('=' * 60)

rows = [
    [log['repo'], log['clone'], log['create'], log['push'], log['delete'],
     log['error'][:40] if log['error'] else '—']
    for log in migration_log
]

headers = ['Repo', 'Clone', 'Create', 'Push', 'Delete', 'Error']
print(tabulate(rows, headers=headers, tablefmt='rounded_outline'))

# Stats
total      = len(migration_log)
successful = sum(1 for l in migration_log if l['push'] == '✅')
failed     = total - successful
deleted    = sum(1 for l in migration_log if l['delete'] == '✅')

print(f'\n  Total repos : {total}')
print(f'  ✅ Migrated : {successful}')
print(f'  ❌ Failed   : {failed}')
print(f'  🗑️  Deleted  : {deleted}')

# Save log to JSON
log_file = os.path.join(CLONE_BASE_DIR, f'migration_log_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json')
os.makedirs(CLONE_BASE_DIR, exist_ok=True)
with open(log_file, 'w') as f:
    json.dump(migration_log, f, indent=2)

print(f'\n💾 Log saved to: {log_file}')

## 🔄 Cell 10 — Migrate a Single Repo (On-Demand)
Use this cell to migrate one specific repo interactively.

In [ ]:
# Change this to the repo name you want to migrate
SINGLE_REPO_NAME = "my-specific-repo"

# Find it in the fetched list
match = next((r for r in repo_data if r['name'] == SINGLE_REPO_NAME), None)

if not match:
    print(f'❌ Repo "{SINGLE_REPO_NAME}" not found in Account 1.')
else:
    print(f'🔄 Migrating single repo: {SINGLE_REPO_NAME}\n')
    try:
        print('  📥 Cloning...')
        clone_path = clone_repo(match, CLONE_BASE_DIR)
        print(f'  ✅ Cloned: {clone_path}')

        print('  🏗️  Creating on Account 2...')
        new_repo = create_repo_on_account2(match)
        print(f'  ✅ Created: {new_repo.html_url}')

        print('  📤 Pushing...')
        push_to_account2(clone_path, match)
        print('  ✅ Pushed successfully!')

        if ALLOW_DELETE:
            print('  🗑️  Deleting from Account 1...')
            delete_from_account1(SINGLE_REPO_NAME)
            print('  ✅ Deleted.')

        shutil.rmtree(clone_path)
        print('  🧹 Cleaned up local clone.')
        print(f'\n✅ Done! Repo is now at: {new_repo.html_url}')

    except Exception as e:
        print(f'❌ Error: {e}')

## 🗑️ Cell 11 — Delete Repos from Account 1 (Standalone)
Use this cell ONLY if you want to delete repos from Account 1 separately (e.g., after verifying the migration).

In [9]:
# Only runs if ALLOW_DELETE is True
ALLOW_DELETE = True  # Set to True to enable deletion
REPOS_TO_DELETE = [
    'ai-interview-question',
    'mastering-projects'
    # Add repo names to delete...
]

if not ALLOW_DELETE:
    print('⚠️  ALLOW_DELETE is False. Set it to True in Cell 2 to enable deletion.')
else:
    print(f'🗑️  About to delete {len(REPOS_TO_DELETE)} repos from Account 1 ({ACCOUNT1_USERNAME})...')
    print('⚠️  THIS IS IRREVERSIBLE!\n')

    for repo_name in REPOS_TO_DELETE:
        print(f'  Deleting {repo_name}...', end=' ')
        deleted = delete_from_account1(repo_name)
        print('✅' if deleted else '❌')

🗑️  About to delete 2 repos from Account 1 (Rajbeer-singh2706)...
⚠️  THIS IS IRREVERSIBLE!

  Deleting ai-interview-question... ✅
  Deleting mastering-projects... ✅


---
## 📖 Quick Reference

| Cell | Purpose |
|------|---------|
| 1 | Install dependencies |
| 2 | 🔑 Set tokens, usernames, clone dir |
| 3 | Connect to GitHub accounts |
| 4 | List all repos from Account 1 |
| 5 | Filter / select repos to migrate |
| 6 | Load helper functions |
| 7 | Dry run preview (no changes) |
| **8** | **▶️ Run full migration** |
| 9 | View summary report + save log |
| 10 | Migrate one specific repo |
| 11 | Delete repos from Account 1 (standalone) |

### Tips
- `git clone --mirror` copies **all branches, tags, and history** (not just main)
- `git push --mirror` pushes everything to the destination
- Keep `ALLOW_DELETE = False` until you've verified the migration in Account 2
- The log JSON file lets you audit what was migrated